In [2]:
%pip install jupyter-black
%load_ext autoreload
%autoreload 2
%load_ext jupyter_black

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 1.1 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
from pathlib import Path
import polars as pl
from IPython.display import display

ModuleNotFoundError: No module named 'polars'

# SRC Data

In [ ]:
data_dir = "/home/chemi_t/Documents/sci_fi/Programming/Job/Ameriabank"
tx_path_str = f"{data_dir}/detail/trx/**/*.parquet"

In [ ]:
! ls {data_dir}/detail/trx/**/*.parquet

In [ ]:
dates = [
    "2020-01-01 13:45:48",
    "2020-01-01 16:42:13",
    "2020-01-01 16:45:09",
    "2020-01-02 18:12:48",
    "2020-01-03 19:45:32",
    "2020-01-08 23:16:43",
]
df = pl.DataFrame({"dt": dates, "a": [3, 7, 5, 9, 2, 1]}).with_columns(
    pl.col("dt").str.strptime(pl.Datetime).set_sorted()
)
df.sort("dt")
lf = df.lazy().sort("dt")
out = lf.rolling(index_column="dt", period="2d").agg(
    [
        pl.sum("a").alias("sum_a"),
        pl.min("a").alias("min_a"),
        pl.max("a").alias("max_a"),
    ]
)

In [ ]:
lf_tx = pl.scan_parquet(tx_path_str)

# trx profiling

In [ ]:
lf_tx.head(n=5).collect()

In [ ]:
max_min_date = lf_tx.select(
    pl.col("event_time").dt.date().max().alias("max_date"),
    pl.col("event_time").dt.date().min().alias("min_date"),
).collect()
date_range = max_min_date["max_date"].item() - max_min_date["min_date"].item()

In [ ]:
date_range

In [ ]:
n_clients = lf_tx.select(pl.col("client_id").n_unique()).collect()
display(n_clients)

In [ ]:
q = lf_tx.select(pl.col("client_id").value_counts())
q = (
    q.collect()
    .select(pl.col("client_id").struct.field("count"))
    .filter(pl.col("count") < 10000)
)
display(q.select(pl.len()))
q["count"].hist(bin_count=10).sort("breakpoint")

# Proto asset ops

In [ ]:
lf_tx_part = lf_tx.filter(pl.col("client_id").cast(pl.String).hash() % 100 == 0)
n_client_in_part = lf_tx_part.select(pl.col("client_id").n_unique()).collect()
n_date_part = lf_tx_part.select(pl.col("event_time").dt.date().n_unique()).collect()
print(n_client_in_part)
print(n_date_part)

In [ ]:
lf_tx_part = lf_tx.filter(pl.col("client_id").cast(pl.String).hash() % 100 == 0)
n_client_in_part = lf_tx_part.select(pl.col("client_id").n_unique())
print(n_client_in_part)
out = (
    lf_tx_part.sort("event_time")
    .select(pl.col("event_time").dt.date(), pl.col("client_id"), pl.col("amount"))
    .rolling(index_column="event_time", period="1w", group_by="client_id")
    .agg(
        [
            pl.sum("amount").alias("sum_a"),
        ]
    )
    .collect()
)

In [ ]:
out.select(pl.col("client_id"), pl.col("event_time").dt.date().alias("date")).group_by(
    "client_id"
).agg(pl.col("date").len())

In [ ]:
date_range.days

In [ ]:
out.select(pl.len())